# Notebook Overview — Prepare Video Data

## Purpose

This notebook prepares the NExT-QA benchmark dataset for iterative Retrieval-Augmented Generation (RAG) Video Question Answering (VideoQA) experimentation. The workflow configures the runtime environment, verifies required dataset resources, reconstructs and extracts video archives, organizes dataset files, validates dataset integrity, and confirms consistency between video, question-answer, and metadata resources required for downstream preprocessing, embedding generation, retrieval, and inference workflows.

## Inputs

* NExT-QA multipart video archive files stored in Google Drive

  * NExTVideo.z01
  * NExTVideo.z02
  * NExTVideo.z03
  * NExTVideo.z04
  * NExTVideo.z05
  * NExTVideo.z06
  * NExTVideo.zip
* NExT-QA question-answer annotation files

  * train.csv
  * val.csv
  * test.csv
* NExT-QA metadata resources

  * map_vid_vidorID.json
* User configuration settings
* Project configuration modules

## Outputs

* Verified NExT-QA dataset directory structure
* Reconstructed and extracted video dataset
* Validated question-answer annotation files
* Verified metadata resources
* Cross-reference validation results
* Random dataset verification samples
* Dataset readiness summary
* Runtime environment configuration information

## Processing Workflow

* Configure runtime environment and project settings
* Mount Google Drive and verify dataset resources
* Copy video archives to local storage
* Reconstruct and extract the NExT-QA video archive
* Validate video, annotation, and metadata resources
* Validate cross-references between videos, question-answer files, and metadata mappings
* Display random dataset verification samples with associated video playback
* Generate a dataset readiness summary

## Notes

* This notebook focuses on dataset preparation, validation, and readiness assessment only.
* The NExT-QA benchmark serves as the primary VideoQA dataset for this project.
* Video archives are copied to local Colab storage and extracted locally to improve reliability and performance.
* Cross-reference validation confirms consistency between video files, question-answer annotations, and metadata mappings.
* Random sample inspection provides visual verification of dataset integrity prior to downstream experimentation.
* Frame extraction, clip generation, embedding creation, vector indexing, retrieval, and inference workflows are performed in later notebooks.
* Video archive reconstruction and extraction may require significant storage space and execution time depending on the runtime environment.


### 🔷 Step 1 — Clone Required Repository Files

* Clone the project repository using sparse checkout to minimize download size and runtime initialization overhead.
* Authenticate access to the private GitHub repository using a fine-grained access token stored in Google Colab Secrets.
* Configure the local notebook workspace and change to the repository working directory.
* Verify that required repository files and directories are available for subsequent notebook execution.
* Optionally display repository paths, directory contents, and cloned files when `VERBOSE=True`.


In [ ]:
# ============================================================
# Step 1: Clone Required Repository Files
# ============================================================

VERBOSE = True

import os
from google.colab import userdata

REPO_NAME = "iterative-video-rag"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(REPO_BASE_DIR, REPO_NAME)

# ------------------------------------------------------------
# Retrieve GitHub Token from Colab Secrets
# ------------------------------------------------------------

github_token = userdata.get("GITHUB_TOKEN")

if github_token is None:
    raise ValueError(
        "GITHUB_TOKEN not found in Colab Secrets."
    )

repo_url = (
    f"https://{github_token}"
    f"@github.com/{REPO_OWNER}/{REPO_NAME}.git"
)

# ------------------------------------------------------------
# Move to Base Directory
# ------------------------------------------------------------

%cd {REPO_BASE_DIR}

# ------------------------------------------------------------
# Clone Repository if Needed
# ------------------------------------------------------------

if not os.path.exists(REPO_DIR):

    if VERBOSE:
        print("Cloning required repository files...")

    !git clone --quiet --filter=blob:none --no-checkout {repo_url}

    %cd {REPO_DIR}

    !git sparse-checkout init --no-cone

    !git sparse-checkout set \
        src/iterative_rag_config.py \
        src/nextqa_video_cache.py \
        datasets/NExT-QA/questions \
        datasets/NExT-QA/metadata

    !git checkout --quiet main

else:

    if VERBOSE:
        print(f"Repository already exists: {REPO_DIR}")

    %cd {REPO_DIR}

# ------------------------------------------------------------
# Verify Repository Setup
# ------------------------------------------------------------

required_paths = [
    "src",
    "src/iterative_rag_config.py",
    "src/nextqa_video_cache.py",
    "datasets/NExT-QA/questions",
    "datasets/NExT-QA/metadata",
]

for path in required_paths:

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Required path not found: {path}"
        )

print("Repository setup complete.")

if VERBOSE:
    print(f"\nCurrent directory: {os.getcwd()}")
    print("\nAvailable files:")
    !find src datasets/NExT-QA -maxdepth 3 -type f | sort


### 🔷 Step 2 — Import Project Configuration

* Import centralized project configuration settings and constants from the project configuration module.
* Load repository paths, dataset directories, and runtime configuration parameters used throughout the VideoQA workflow.
* Initialize reusable configuration values shared across notebook execution stages.
* Verify that required configuration resources are accessible and correctly loaded before continuing.
* Optionally display configuration settings and resolved paths when `VERBOSE=True`.

In [ ]:
# ============================================================
# Step 2: Import Project Configuration
# ============================================================

# ------------------------------------------------------------
# Import Centralized Project Configuration Values
# ------------------------------------------------------------

try:

    from src.iterative_rag_config import *

except ModuleNotFoundError as error:

    raise ModuleNotFoundError(
        "Project configuration could not be imported. "
        "Verify that the repository was cloned correctly and that "
        "'src/iterative_rag_config.py' exists."
    ) from error


# ------------------------------------------------------------
# Import NExT-QA Video Cache Utilities
# ------------------------------------------------------------

try:

    from src.nextqa_video_cache import (
        verify_nextqa_archive_parts,
        copy_nextqa_archive_parts_to_local,
        build_combined_nextqa_archive,
        extract_nextqa_video_archive,
        verify_nextqa_video_cache,
    )

except ModuleNotFoundError as error:

    raise ModuleNotFoundError(
        "NExT-QA video cache utilities could not be imported. "
        "Verify that 'src/nextqa_video_cache.py' exists."
    ) from error


# ------------------------------------------------------------
# Verify Required Configuration Values
# ------------------------------------------------------------

required_config_values = [
    "BASE_DIR",
    "DATASETS_DIR",
    "DATASET_CONFIG",
]

missing_config_values = [
    name
    for name in required_config_values
    if name not in globals()
]

if missing_config_values:

    raise ValueError(
        "Missing required configuration values: "
        + ", ".join(missing_config_values)
    )


# ------------------------------------------------------------
# Display Configuration Summary
# ------------------------------------------------------------

print("Project configuration and video cache utilities imported successfully.")

if VERBOSE:
    print(f"BASE_DIR:      {BASE_DIR}")
    print(f"DATASETS_DIR:  {DATASETS_DIR}")

    print("\nConfigured Datasets:")

    for dataset_name in DATASET_CONFIG:
        print(f"  - {dataset_name}")


### 🔷 Step 3 — Mount Google Drive

* Mount Google Drive to access benchmark dataset archives and supporting resources.
* Verify that Google Drive was mounted successfully and is accessible from the notebook runtime.
* Configure access to the NExT-QA dataset storage location.
* Confirm that required dataset resources are available before proceeding with extraction and validation steps.
* Optionally display mounted paths and available dataset files when `VERBOSE=True`.

In [ ]:
# ============================================================
# Step 3: Mount Google Drive
# ============================================================

# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------

from google.colab import drive
import os

GOOGLE_DRIVE_MOUNT = "/content/drive"

if not os.path.exists(GOOGLE_DRIVE_MOUNT):

    if VERBOSE:
        print("Mounting Google Drive...")

    drive.mount(GOOGLE_DRIVE_MOUNT)

else:

    if VERBOSE:
        print("Google Drive is already mounted.")


# ------------------------------------------------------------
# Verify Google Drive Access
# ------------------------------------------------------------

if not os.path.exists(GOOGLE_DRIVE_MOUNT):

    raise FileNotFoundError(
        "Google Drive mount point was not found."
    )

drive_root = os.path.join(
    GOOGLE_DRIVE_MOUNT,
    "MyDrive"
)

if not os.path.exists(drive_root):

    raise FileNotFoundError(
        "Unable to access Google Drive root directory."
    )

print("Google Drive mounted successfully.")


# ------------------------------------------------------------
# Display Drive Information
# ------------------------------------------------------------

if VERBOSE:
    print(f"\nDrive Root: {drive_root}")
    print("\nTop-Level Google Drive Folders:")

    try:
        drive_items = sorted(os.listdir(drive_root))
        for item in drive_items[:20]:
            print(f"  {item}")
        if len(drive_items) > 20:
            print(
                f"\n... and "
                f"{len(drive_items) - 20} additional items"
            )

    except Exception as error:

        print(
            f"Unable to list Google Drive contents: {error}"
        )



### 🔷 Step 4 — Verify NExT-QA Dataset Resources

* Configure the Google Drive location containing the NExT-QA dataset archive files.
* Verify that all required multipart video archive files are present.
* Display file sizes for the available dataset resources.
* Stop notebook execution if any required archive files are missing.
* Prepare validated archive paths for later extraction.

In [ ]:
# ============================================================
# Step 4: Verify NExT-QA Dataset Resources
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# Configure Google Drive Dataset Source Directory
# ------------------------------------------------------------

DRIVE_DATASET_DIR = Path(drive_root) / "VideoQA_Project" / "NExT-QA"

# ------------------------------------------------------------
# Define Required Archive Files
# ------------------------------------------------------------

required_archive_files = [
    "NExTVideo.z01",
    "NExTVideo.z02",
    "NExTVideo.z03",
    "NExTVideo.z04",
    "NExTVideo.z05",
    "NExTVideo.z06",
    "NExTVideo.zip",
]

# ------------------------------------------------------------
# Verify Required Archive Files
# ------------------------------------------------------------

archive_verification_summary = verify_nextqa_archive_parts(
    archive_parts_dir=DRIVE_DATASET_DIR,
    required_archive_files=required_archive_files,
    verbose=VERBOSE,
)

# ------------------------------------------------------------
# Display Available Dataset Files
# ------------------------------------------------------------

if VERBOSE:

    print("\nAvailable files:")

    for file_path in sorted(DRIVE_DATASET_DIR.iterdir()):

        if file_path.is_file():

            file_size_mb = file_path.stat().st_size / (1024 ** 2)

            print(
                f"  {file_path.name:<28} "
                f"{file_size_mb:10.2f} MB"
            )


### 🔷 Step 5 — Copy NExT-QA Archive Files to Local Storage

* Create a local archive workspace within the Colab runtime environment.
* Copy the NExT-QA multipart archive files from Google Drive to local storage.
* Verify that all required archive files were copied successfully.
* Display archive file sizes and storage utilization information when `VERBOSE=True`.
* Skip file copies when valid local archive files already exist.
* Prepare local archive resources for archive reconstruction and extraction in later steps.

In [ ]:
# ============================================================
# Step 5: Copy NExT-QA Archive Files to Local Storage
# ============================================================

# ------------------------------------------------------------
# Configure Local Archive Directory
# ------------------------------------------------------------

LOCAL_ARCHIVE_DIR = DATASETS_DIR / "NExT-QA" / "archives"

# ------------------------------------------------------------
# Copy Archive Files to Local Storage
# ------------------------------------------------------------

local_archive_summary = copy_nextqa_archive_parts_to_local(
    source_archive_dir=DRIVE_DATASET_DIR,
    local_archive_dir=LOCAL_ARCHIVE_DIR,
    required_archive_files=required_archive_files,
    verbose=VERBOSE,
)


### 🔷 Step 6 — Build Combined NExT-QA Archive

* Reconstruct the original NExT-QA video archive from the locally copied multipart archive files.
* Concatenate all archive segments in the correct sequence to create a single combined ZIP archive.
* Verify that the combined archive was created successfully and has a valid file size.
* Skip archive reconstruction when a valid combined archive already exists.
* Display archive creation statistics and storage utilization information when `VERBOSE=True`.
* Prepare the combined archive for local extraction in the next processing step.


In [ ]:
# ============================================================
# Step 6: Build Combined NExT-QA Archive
# ============================================================

# ------------------------------------------------------------
# Configure Combined Archive Path
# ------------------------------------------------------------

COMBINED_ARCHIVE_PATH = LOCAL_ARCHIVE_DIR / "NExTVideo_combined.zip"

# ------------------------------------------------------------
# Build Combined Archive
# ------------------------------------------------------------

combined_archive_summary = build_combined_nextqa_archive(
    local_archive_dir=LOCAL_ARCHIVE_DIR,
    combined_archive_path=COMBINED_ARCHIVE_PATH,
    split_archive_name="NExTVideo.zip",
    required_archive_files=required_archive_files,
    force_rebuild=True,
    verbose=VERBOSE,
)


### 🔷 Step 7 — Extract NExT-QA Video Archive

* Create the local NExT-QA video directory within the project dataset workspace.
* Extract the combined NExT-QA video archive into local Colab storage.
* Preserve the original NExT-QA video folder structure during extraction.
* Skip extraction when extracted video files are already present.
* Verify that extraction completed without archive errors.
* Prepare the extracted video dataset for structure validation in the next step.


In [ ]:
# ============================================================
# Step 7: Extract NExT-QA Video Archive
# ============================================================

# ------------------------------------------------------------
# Configure Local Video Directory
# ------------------------------------------------------------

NEXTQA_DATASET_DIR = DATASETS_DIR / "NExT-QA"
NEXTQA_VIDEOS_DIR = NEXTQA_DATASET_DIR / "videos"

# ------------------------------------------------------------
# Extract Combined Archive and Verify Video Cache
# ------------------------------------------------------------

video_cache_summary = extract_nextqa_video_archive(
    combined_archive_path=COMBINED_ARCHIVE_PATH,
    local_videos_dir=NEXTQA_VIDEOS_DIR,
    force_extract=False,
    verbose=VERBOSE,
)


### 🔷 Step 8 — Verify Question and Metadata Resources

* Verify the presence of required NExT-QA question-answer annotation files.
* Verify the presence of required NExT-QA metadata resources.
* Validate accessibility of training, validation, and test dataset files.
* Display annotation file sizes and basic dataset statistics.
* Confirm that all required resources are available for downstream VideoQA experimentation.
* Stop notebook execution if required annotation or metadata files are missing.


In [ ]:
# ============================================================
# Step 8: Verify Question and Metadata Resources
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# Configure NExT-QA Dataset Resource Directories
# ------------------------------------------------------------

#NEXTQA_DATASET_DIR = DATASETS_DIR / "NExT-QA"
NEXTQA_QUESTIONS_DIR = DATASET_CONFIG["NExT-QA"]["questions_dir"]
NEXTQA_METADATA_DIR  = DATASET_CONFIG["NExT-QA"]["metadata_dir"]

NEXTQA_QUESTIONS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

NEXTQA_METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# Define Required Question and Metadata Files
# ------------------------------------------------------------

required_question_files = [
    "train.csv",
    "val.csv",
    "test.csv",
]

required_metadata_files = [
    "map_vid_vidorID.json",
]

# ------------------------------------------------------------
# Verify Required Files
# ------------------------------------------------------------

missing_question_files = []
missing_metadata_files = []

for filename in required_question_files:

    file_path = NEXTQA_QUESTIONS_DIR / filename

    if not file_path.exists():

        missing_question_files.append(filename)

for filename in required_metadata_files:

    file_path = NEXTQA_METADATA_DIR / filename

    if not file_path.exists():

        missing_metadata_files.append(filename)

if missing_question_files:

    raise FileNotFoundError(
        "Missing required question files: "
        + ", ".join(missing_question_files)
    )

if missing_metadata_files:

    raise FileNotFoundError(
        "Missing required metadata files: "
        + ", ".join(missing_metadata_files)
    )

# ------------------------------------------------------------
# Load Question Files and Count Rows
# ------------------------------------------------------------

question_counts = {}

for filename in required_question_files:

    file_path = NEXTQA_QUESTIONS_DIR / filename

    dataframe = pd.read_csv(file_path)

    question_counts[filename] = len(dataframe)

# ------------------------------------------------------------
# Display Verification Summary
# ------------------------------------------------------------

print("NExT-QA question and metadata resources verified.")

print("\nQuestion Files:")

for filename in required_question_files:

    file_path = NEXTQA_QUESTIONS_DIR / filename
    file_size_mb = file_path.stat().st_size / (1024 ** 2)

    print(
        f"  {filename:<10} "
        f"{question_counts[filename]:>8} rows "
        f"{file_size_mb:>8.2f} MB"
    )

print("\nMetadata Files:")

for filename in required_metadata_files:

    file_path = NEXTQA_METADATA_DIR / filename
    file_size_mb = file_path.stat().st_size / (1024 ** 2)

    print(
        f"  {filename:<32} "
        f"{file_size_mb:>8.2f} MB"
    )

# ------------------------------------------------------------
# Display Optional Dataset Details
# ------------------------------------------------------------

if VERBOSE:

    total_questions = sum(question_counts.values())

    print(f"\nTotal Questions: {total_questions}")

    print("\nDataset Resource Directories:")
    print(f"  Questions: {NEXTQA_QUESTIONS_DIR}")
    print(f"  Metadata:  {NEXTQA_METADATA_DIR}")



### 🔷 Step 9 — Validate Dataset Cross-References

* Verify that video IDs referenced in the NExT-QA question-answer files map to extracted video files.
* Verify that video IDs referenced in the question-answer files are present in the metadata mappings.
* Validate dataset consistency across video, question-answer, and metadata resources.
* Report video coverage, metadata coverage, and any unresolved references.
* Confirm that all required dataset relationships are valid prior to sample inspection and downstream processing.


In [ ]:
# ============================================================
# Step 9: Validate Dataset Cross-References
# ============================================================

import json
import pandas as pd

# ------------------------------------------------------------
# Configure NExT-QA Dataset Resource Directories
# ------------------------------------------------------------

NEXTQA_CONFIG = DATASET_CONFIG["NExT-QA"]

NEXTQA_VIDEOS_DIR = NEXTQA_CONFIG["videos_dir"]
NEXTQA_QUESTIONS_DIR = NEXTQA_CONFIG["questions_dir"]
NEXTQA_METADATA_DIR = NEXTQA_CONFIG["metadata_dir"]

# ------------------------------------------------------------
# Load Question-Answer Files
# ------------------------------------------------------------

question_files = {
    "train": NEXTQA_QUESTIONS_DIR / "train.csv",
    "val": NEXTQA_QUESTIONS_DIR / "val.csv",
    "test": NEXTQA_QUESTIONS_DIR / "test.csv",
}

qa_dataframes = {}

for split_name, file_path in question_files.items():

    qa_dataframes[split_name] = pd.read_csv(file_path)

# ------------------------------------------------------------
# Identify Video ID Column
# ------------------------------------------------------------

candidate_video_columns = [
    "video",
    "video_id",
    "vid",
    "video_name",
]

video_column = None

for column_name in candidate_video_columns:

    if column_name in qa_dataframes["train"].columns:

        video_column = column_name
        break

if video_column is None:

    raise ValueError(
        "Could not identify the video ID column "
        "in the QA files."
    )

# ------------------------------------------------------------
# Collect Referenced Video IDs
# ------------------------------------------------------------

qa_video_ids_by_split = {}

for split_name, dataframe in qa_dataframes.items():

    qa_video_ids_by_split[split_name] = set(
        dataframe[video_column]
        .astype(str)
        .str.replace(".mp4", "", regex=False)
    )

all_qa_video_ids = set().union(
    *qa_video_ids_by_split.values()
)

# ------------------------------------------------------------
# Collect Available Video Files
# ------------------------------------------------------------

video_files = sorted(
    NEXTQA_VIDEOS_DIR.rglob("*.mp4")
)

video_file_lookup = {
    file_path.stem: file_path
    for file_path in video_files
}

available_video_ids = set(
    video_file_lookup.keys()
)

# ------------------------------------------------------------
# Validate QA References Against Video Files
# ------------------------------------------------------------

missing_video_ids = sorted(
    all_qa_video_ids - available_video_ids
)

matched_video_ids = sorted(
    all_qa_video_ids & available_video_ids
)

# ------------------------------------------------------------
# Load Metadata Mapping File
# ------------------------------------------------------------

mapping_file = (
    NEXTQA_METADATA_DIR /
    "map_vid_vidorID.json"
)

with open(mapping_file, "r") as file:

    video_id_mapping = json.load(file)

metadata_video_ids = set(
    str(video_id).replace(".mp4", "")
    for video_id in video_id_mapping.keys()
)

# ------------------------------------------------------------
# Validate QA References Against Metadata
# ------------------------------------------------------------

missing_metadata_ids = sorted(
    all_qa_video_ids - metadata_video_ids
)

matched_metadata_ids = sorted(
    all_qa_video_ids & metadata_video_ids
)

# ------------------------------------------------------------
# Build Combined QA DataFrame
# ------------------------------------------------------------

combined_qa_dataframe = pd.concat(
    [
        dataframe.assign(split=split_name)
        for split_name, dataframe
        in qa_dataframes.items()
    ],
    ignore_index=True,
)

combined_qa_dataframe["_video_id_normalized"] = (
    combined_qa_dataframe[video_column]
    .astype(str)
    .str.replace(".mp4", "", regex=False)
)

# ------------------------------------------------------------
# Display Cross-Reference Summary
# ------------------------------------------------------------

print("NExT-QA Dataset Cross-Reference Validation")
print("=" * 60)

print("\nQuestion-Answer Video References")
print("-" * 60)

for split_name, video_ids in qa_video_ids_by_split.items():

    print(
        f"{split_name:<8}: "
        f"{len(video_ids):>6} unique referenced videos"
    )

print(
    f"{'Total':<8}: "
    f"{len(all_qa_video_ids):>6} unique referenced videos"
)

print("\nQA to Video File Coverage")
print("-" * 60)

print(
    f"Referenced QA video IDs : "
    f"{len(all_qa_video_ids):>6}"
)

print(
    f"Available video files   : "
    f"{len(available_video_ids):>6}"
)

print(
    f"Matched video files     : "
    f"{len(matched_video_ids):>6}"
)

print(
    f"Missing video files     : "
    f"{len(missing_video_ids):>6}"
)

print("\nQA to Metadata Coverage")
print("-" * 60)

print(
    f"Metadata mapping entries : "
    f"{len(metadata_video_ids):>6}"
)

print(
    f"Matched metadata IDs     : "
    f"{len(matched_metadata_ids):>6}"
)

print(
    f"Missing metadata IDs     : "
    f"{len(missing_metadata_ids):>6}"
)

# ------------------------------------------------------------
# Display Missing References
# ------------------------------------------------------------

if missing_video_ids:

    print("\nMissing Video File References")
    print("-" * 60)

    for video_id in missing_video_ids[:20]:

        print(f"  {video_id}")

if missing_metadata_ids:

    print("\nMissing Metadata References")
    print("-" * 60)

    for video_id in missing_metadata_ids[:20]:

        print(f"  {video_id}")

# ------------------------------------------------------------
# Final Validation Status
# ------------------------------------------------------------

print("\nCross-Reference Validation Status")
print("-" * 60)

if (
    len(missing_video_ids) == 0
    and
    len(missing_metadata_ids) == 0
):

    print("Status : PASSED")

    print(
        "All question-answer references resolve "
        "to video files and metadata mappings."
    )

else:

    print("Status : REVIEW REQUIRED")

    print(
        "One or more references could not be "
        "resolved."
    )



### 🔷 Step 10 — Display Random Dataset Verification Samples

* Select a small random sample of validated NExT-QA records.
* Resolve answer-choice indices to their corresponding answer text.
* Display representative question-answer pairs alongside associated video metadata.
* Display embedded video playback for visual inspection and sanity checking.
* Verify that videos, questions, answers, and metadata appear consistent and correctly linked.


In [ ]:
# ============================================================
# Step 10: Display Random Dataset Verification Samples
# ============================================================

import random

from IPython.display import Video, display

# ------------------------------------------------------------
# Configure Sample Display Settings
# ------------------------------------------------------------

sample_size = min(
    2,
    len(matched_video_ids),
)

sample_video_ids = random.sample(
    matched_video_ids,
    sample_size,
)

# ------------------------------------------------------------
# Display Random Samples with QA Context and Video Playback
# ------------------------------------------------------------

print("Random NExT-QA Dataset Verification Samples")
print("=" * 60)

for index, video_id in enumerate(sample_video_ids, start=1):

    sample_rows = combined_qa_dataframe[
        combined_qa_dataframe["_video_id_normalized"] == video_id
    ]

    sample_row = sample_rows.sample(
        n=1,
        random_state=index,
    ).iloc[0]

    video_path = video_file_lookup[video_id]

    metadata_value = video_id_mapping.get(
        video_id,
        "NOT FOUND",
    )

    print(f"\nSample {index}")
    print("-" * 60)
    print(f"Split        : {sample_row['split']}")
    print(f"Video ID     : {video_id}")
    print(f"Video File   : {video_path.name}")
    print(f"Metadata Map : {video_id} -> {metadata_value}")

    if "type" in sample_row:
        print(f"Question Type: {sample_row['type']}")

    if "question" in sample_row:
        print(f"Question     : {sample_row['question']}")

    if "answer" in sample_row:

        answer_id = int(sample_row["answer"])
        answer_column = f"a{answer_id}"

        print(f"Answer ID    : {answer_id}")

        if answer_column in sample_row:
            print(f"Answer Text  : {sample_row[answer_column]}")
        else:
            print("Answer Text  : NOT FOUND")

    print("\nAnswer Choices")

    for choice_index in range(5):

        choice_column = f"a{choice_index}"

        if choice_column in sample_row:

            choice_text = sample_row[choice_column]

            if (
                "answer" in sample_row
                and
                choice_index == int(sample_row["answer"])
            ):
                print(f"  {choice_column}: {choice_text}  <-- correct")
            else:
                print(f"  {choice_column}: {choice_text}")

    display(
        Video(
            str(video_path),
            embed=True,
            width=480,
        )
    )

    print("Status       : VERIFIED")



### 🔷 Step 11 — Display Dataset Readiness Summary

* Summarize the results of dataset resource verification, extraction, and validation.
* Confirm that video, question-answer, and metadata resources were successfully processed.
* Confirm that cross-reference validation completed without unresolved references.
* Confirm that random sample inspection and video playback verification were successful.
* Report overall dataset readiness for video knowledge-base construction and downstream VideoQA experimentation.


In [ ]:
# ============================================================
# Step 11: Display Dataset Readiness Summary
# ============================================================

print("=" * 35)
print("NExT-QA Dataset Readiness Summary")
print("=" * 35)

# ------------------------------------------------------------
# Resource Verification Status
# ------------------------------------------------------------

video_resource_status = (
    "PASSED"
    if len(available_video_ids) > 0
    else "FAILED"
)

qa_resource_status = (
    "PASSED"
    if all(
        len(dataframe) > 0
        for dataframe in qa_dataframes.values()
    )
    else "FAILED"
)

metadata_resource_status = (
    "PASSED"
    if len(metadata_video_ids) > 0
    else "FAILED"
)

cross_reference_status = (
    "PASSED"
    if (
        len(missing_video_ids) == 0
        and
        len(missing_metadata_ids) == 0
    )
    else "FAILED"
)

sample_validation_status = (
    "PASSED"
    if len(sample_video_ids) > 0
    else "FAILED"
)

# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print(f"Video Resources            : {video_resource_status}")
print(f"Question-Answer Resources  : {qa_resource_status}")
print(f"Metadata Resources         : {metadata_resource_status}")
print(f"Cross-Reference Validation : {cross_reference_status}")
print(f"Sample Verification        : {sample_validation_status}")

# ------------------------------------------------------------
# Overall Readiness Assessment
# ------------------------------------------------------------

overall_status = all([
    video_resource_status == "PASSED",
    qa_resource_status == "PASSED",
    metadata_resource_status == "PASSED",
    cross_reference_status == "PASSED",
    sample_validation_status == "PASSED",
])

